## Running an Agent in a loop to accomplish a Task

In [52]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from rich.console import Console
load_dotenv(override=True)


True

In [53]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


In [54]:

openai = OpenAI()

In [55]:
todos= []
completed = []

In [56]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result  

In [57]:
get_todo_report()

''

In [78]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [79]:
# Now complete the task
def mark_complete(index: int, completion_notes: str):
    if 1 <= index <= len(todos):
        completed [index - 1] = True
    else:
        return "No todo at this postion."
    Console().print(completion_notes)
    return get_todo_report()    

In [ ]:
todos, completed = [], []
create_todos(["Attend Seminar", "Read book", "Eat breakfast"])

In [ ]:
mark_complete(1, "I attended the seminar")

In [82]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [83]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [84]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [85]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name =  tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)    # important to note
        result = tool(**arguments) if tool else {}   # clever
        results.append({"role": "tool", "content" : json.dumps(result), "tool_call_id": tool_call.id})
    return results    


In [86]:
# for gpt-5-non or gpt-4* the reasoning_effort=None is not suppported instead it supports values: minimal/low etc
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model = "gpt-5-nano", messages=messages, tools=tools, reasoning_effort="minimal")
        finish_reason = response.choices[0].finish_reason
        print(finish_reason)
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)            


In [87]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
You are the lead Identity Governance Engineer for a multinational bank that has recently acquired another global bank. 
The merged organization now has 50,000 employees, 300 applications, 20 Active Directory domains, multiple cloud platforms, 
and operations across several countries. 
The CIO has tasked you with completing a 90-day identity governance transformation program to ensure all users have appropriate access,
 eliminate toxic Segregation of Duties (SoD) conflicts, certify privileged access, identify and remediate orphan accounts, 
 consolidate roles and entitlements, validate Joiner-Mover-Leaver processes, prepare audit evidence for SOX and GDPR compliance, 
 and decommission legacy identities and systems. Create, track, prioritize, and complete all tasks required to successfully execute 
 this project while maintaining regulatory compliance and minimizing operational risk.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loop(messages)